In [ ]:
import psutil
nthreads = psutil.cpu_count(logical=False)
# nthreads = 1

In [ ]:
print(f"Num of threads using: ", nthreads)

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(nthreads)
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [ ]:
os.getpid()

In [ ]:
import sys; sys.path.append('..')
sys.path.append('../curved_linesearch/')
import MeshFEM, mesh, mesh_energy, benchmark, viewer, py_newton_optimizer
import differential_operators

import numpy as np
import igl
import copy, time

import matplotlib
from matplotlib import pyplot as plt

In [ ]:
import sim_utils, param_utils
import extra_utils, opt_utils

In [ ]:
import newton_flow
import newton_flow_utils as nfu

In [ ]:
from Benchmark import helper_funcs

In [ ]:
import parallelism

# Construct Newton Flow as Param

In [ ]:
model = 'Hilbert2.off'
model_name = 'Hilbert'
initial_uv_mesh = 'ToysMesh/Hilbert_init_2d.obj'

In [ ]:
m = helper_funcs.read_mesh(f'../../../Models/TableOneModels/{model}')
print(f"Model: {model} Vertices: {m.numVertices()}")
print(f"Model: {model} Elements: {m.numElements()}")

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
m_2d = mesh.Mesh(np.zeros((m.numVertices(),2)), m.elements())
m_2d.reembedElements(m.vertices())


m_init_2d = mesh.Mesh(initial_uv_mesh)
uv.setVars(m_init_2d.vertices().ravel())
nf = newton_flow.symmetric_dirichlet(m_2d, uv)

In [ ]:
# Construct NewtonMultiObjective Problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [nf])

# nf, prob, and opt settings

In [ ]:
nf.elementHessianShift = 1e-9
prob.hessianShift = 0
prob.useRelativeHessianShift = False

In [ ]:
FIX_VARS = False
always_project = False

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.options.niter = 200

In [ ]:
prob.energy()

# Extrapolator Construction and Linesearch routine

In [ ]:
import rotation_strain_extrapolation

In [ ]:
RS_extrapolator = rotation_strain_extrapolation.RSNewtonFlowExtrapolator(m_2d)

In [ ]:
brutal_line_search = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)
ternary_line_search = opt_utils.TernaryLinesearch()
golden_section_search = opt_utils.GoldenSectionSearch()
parabola_fit_search = opt_utils.ParabolaFitSearch()

In [ ]:
max_alpha = 10
parabola_fit_search.max_alpha = max_alpha
parabola_fit_search.use_extrapolation = True

In [ ]:
line_search_method = parabola_fit_search

# Customer Callback

In [ ]:
SAVE_UV_FILES = True

In [ ]:
base_folder = 'IterTimeData/HilbertTestOnMac_0421'

In [ ]:
# method_name = 'ord_newton'
method_name = 'RS'

In [ ]:
if SAVE_UV_FILES: work_path = os.path.join(base_folder, model_name, method_name, 'UVs')
else:             work_path = os.path.join(base_folder, model_name, method_name, f'thread_{nthreads}')

os.makedirs(work_path, exist_ok=True)

In [ ]:
obj_history = []
grad_norm_history = []
time_history = []

globar_ind = 0

In [ ]:
def customCallback(prob, i):
    obj_history.append(prob.energy())
    grad_norm_history.append(np.linalg.norm(prob.gradient()))
    time_history.append(-benchmark.totalTime('Callback$') + time.perf_counter())

def customSaveUVCallback(prob, i):
    obj_history.append(prob.energy())
    grad_norm_history.append(np.linalg.norm(prob.gradient()))
    
    # Save UV in compressed mode
    uv_fn = 'uv_ravel_' + 'iter_' + str(i + globar_ind - 1)
    uv_arr = uv.getVars()
    np.savez_compressed(os.path.join(work_path, uv_fn), arr=uv_arr)
    time_history.append(-benchmark.totalTime('Callback$') + time.perf_counter())

In [ ]:
if SAVE_UV_FILES:
    if not os.path.exists(work_path):
        raise RuntimeError(f"[Error] The uv_save path: {work_path} does not exist!")
    prob.setCustomIterationCallback(customSaveUVCallback)
else:  prob.setCustomIterationCallback(customCallback)

# Optimize

In [ ]:
max_iters = 15

In [ ]:
opt.options.niter = max_iters
opt.options.gradTol = 2e-8

In [ ]:
benchmark.reset()
start_time = time.perf_counter()

if method_name == 'ord_newton':
    opt.options.niter = 200
    cr = opt.optimize()
elif method_name == 'RS':
    if SAVE_UV_FILES:  custom_pre_cb = customSaveUVCallback
    else:              custom_pre_cb = customCallback
    
    cr = opt.optimize()
    globar_ind += len(cr.energy)
    vertices_list = opt_utils.newton_extrapolate(opt, RS_extrapolator, line_search_method, grad_tol=opt.options.gradTol, max_iters=50, 
                                                 pre_step_cb = custom_pre_cb,
                                                 verbose=False, newton_step_tol=1e-4, max_extraNewton_stop_counter=1)
    globar_ind += len(vertices_list) - 1
    opt.optimize()

benchmark.report()

In [ ]:
len(time_history)

In [ ]:
len(grad_norm_history)

In [ ]:
time_history[16] - time_history[15]

In [ ]:
time_history[15] - time_history[14]

In [ ]:
time_history[17] - time_history[16]

# Save Exp Data

In [ ]:
# brek

In [ ]:
bk_dict = benchmark.to_dict()
time_arr = np.array(time_history) - start_time
obj_arr = np.array(obj_history)
grad_norm_arr = np.array(grad_norm_history)

In [ ]:
obj_filename = 'obj_history.npy'
time_filename = 'time_history.npy'
grad_norm_filename = 'grad_norm_history.npy'
benchmark_filename = 'benchmark_dict.pkl'

np.save(os.path.join(work_path, obj_filename), obj_arr)
np.save(os.path.join(work_path, time_filename), time_arr)
np.save(os.path.join(work_path, grad_norm_filename), grad_norm_arr)

helper_funcs.save_dict(bk_dict, os.path.join(work_path, benchmark_filename))
print(f"[File] Saved Exp '.npy' files, {obj_filename}, {time_filename}, {grad_norm_filename}, {benchmark_filename} in {work_path}.")

# Visualization matplot

In [ ]:
# brek

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
fig = plt.figure(figsize=(6, 5))
plt.title(f'Model - {model_name}', fontsize=18)
plt.semilogy(grad_norm_arr)
plt.xlabel('Iteration', fontsize=14)
plt.ylabel('Grad Norm', fontsize=16)

In [ ]:
fig = plt.figure(figsize=(6, 5))
plt.title(f'Model - {model_name}', fontsize=18)
plt.plot(time_arr, grad_norm_arr)
plt.yscale('log')
plt.xlabel('Time [s]', fontsize=14)
plt.ylabel('Grad Norm', fontsize=16)